In [ ]:
# Install dependencies and connect Google Drive
# Install the latest Hugging Face libraries
!pip install -q transformers datasets accelerate tokenizers

from google.colab import drive
import os

# Mount Drive to save files securely
drive.mount('/content/drive')

# Create the project directory in Drive if it does not exist
PROJECT_DIR = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Working directory ready at: {PROJECT_DIR}")

Mounted at /content/drive
Diretório de trabalho pronto em: /content/drive/MyDrive/Colab_LLMs/Maia_Lite


In [ ]:
# Install dependencies and connect Google Drive
# Install the latest Hugging Face libraries
!pip install -q transformers datasets accelerate tokenizers

from google.colab import drive
import os

# Mount Drive to save files securely
drive.mount('/content/drive')

# Create the project directory in Drive if it does not exist
PROJECT_DIR = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Working directory ready at: {PROJECT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Diretório de trabalho pronto em: /content/drive/MyDrive/Colab_LLMs/Maia_Lite


## Hugging Face Hub Token Configuration

To avoid unauthenticated request warnings and potentially speed up model and dataset downloads, it is recommended to configure a Hugging Face Hub access token. Follow the steps below:

1.  **Get your Access Token:**
    *   Go to [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
    *   Create a new token. A token with `read` permission is recommended for most download operations.

2.  **Add the Token to Colab Secrets:**
    *   In the left panel of Google Colab, click on the key icon (🔑) to open the "Secrets" interface.
    *   Click on "Add new secret".
    *   In the "Name" field, type `HF_TOKEN`.
    *   In the "Value" field, paste the token you got from Hugging Face.
    *   Make sure to enable the "Notebook access" toggle so the notebook can use this secret.

3.  **Run the Python cell below:**
    *   This cell will load the token from Colab secrets and set it as an environment variable, which will be used by Hugging Face libraries.

In [ ]:
# Import the necessary libraries
from google.colab import userdata
import os
from huggingface_hub import login # Imports the login function from Hugging Face Hub

# Load the Hugging Face token from Colab secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        # Try to login with the Hugging Face token
        login(token=hf_token, add_to_git_credential=False) # 'add_to_git_credential=False' to avoid asking for git credentials
        print("Hugging Face token successfully loaded and configured via huggingface_hub.login().")
    else:
        print("WARNING: The secret 'HF_TOKEN' was found, but it is empty or None. Please check the value in Colab secrets.")
        # If the token is empty/None, still set the environment variable as an empty string to avoid later errors
        os.environ['HF_TOKEN'] = ''
except userdata.SecretNotFoundError:
    print("WARNING: The secret 'HF_TOKEN' was not found. Please add your Hugging Face token to Colab secrets.")
    os.environ['HF_TOKEN'] = '' # Ensure the environment variable is set, even if empty
except Exception as e:
    print(f"An error occurred while loading or configuring the Hugging Face token: {e}")
    os.environ['HF_TOKEN'] = '' # Ensure the environment variable is set, even if empty

Token do Hugging Face carregado e configurado com sucesso via huggingface_hub.login().


In [ ]:
from datasets import load_dataset, concatenate_datasets
from tokenizers import ByteLevelBPETokenizer
from transformers import GPT2TokenizerFast
import os
import shutil

PROJECT_DIR = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
tokenizer_dir = os.path.join(PROJECT_DIR, "tokenizador")

# Checks if the tokenizer already exists and loads it, otherwise trains a new one.
if os.path.exists(tokenizer_dir) and os.path.isdir(tokenizer_dir) and \
   os.path.exists(os.path.join(tokenizer_dir, "vocab.json")) and \
   os.path.exists(os.path.join(tokenizer_dir, "merges.txt")):
    print(f"Tokenizer found at: {tokenizer_dir}. Loading existing tokenizer...")
    tokenizer = GPT2TokenizerFast.from_pretrained(tokenizer_dir, local_files_only=True)
    print("Tokenizer successfully loaded!")
else:
    print(f"Tokenizer not found or incomplete at {tokenizer_dir}. Training a new tokenizer...")

    # 1. Load exact fractions directly to the local Colab cache (WITHOUT streaming=True)
    # Divided proportionally to sum up to 50,000 articles in total (50% EN, 25% PT, 25% ES)
    print("Downloading Wikipedia slices to local memory (This takes about 1-2 minutes)...")
    wiki_en = load_dataset("wikimedia/wikipedia", "20231101.en", split="train[:25000]")
    wiki_pt = load_dataset("wikimedia/wikipedia", "20231101.pt", split="train[:12500]")
    wiki_es = load_dataset("wikimedia/wikipedia", "20231101.es", split="train[:12500]")

    # Merges the locally downloaded datasets and shuffles in RAM
    print("Mixing and preparing the data...")
    mixed_dataset = concatenate_datasets([wiki_en, wiki_pt, wiki_es])
    mixed_dataset = mixed_dataset.shuffle(seed=42)

    # Generator used to feed the tokenizer trainer directly from RAM
    def extract_text():
        for item in mixed_dataset:
            yield item["text"]

    print("Training the tokenizer... This should now take 2 to 3 minutes.")
    tokenizer_raw = ByteLevelBPETokenizer()
    tokenizer_raw.train_from_iterator(
        extract_text(),
        vocab_size=50257, # Classic GPT-2 default
        min_frequency=2,
        special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"]
    )

    # Saves the tokenizer to Drive
    # REMOVES the existing directory before creating again to force synchronization
    if os.path.exists(tokenizer_dir) and os.path.isdir(tokenizer_dir):
        print(f"Removing existing tokenizer directory: {tokenizer_dir}")
        shutil.rmtree(tokenizer_dir)

    os.makedirs(tokenizer_dir, exist_ok=True) # Creates the directory again
    tokenizer_raw.save_model(tokenizer_dir)

    # Converts to the format usable by Hugging Face Trainer
    tokenizer = GPT2TokenizerFast.from_pretrained(tokenizer_dir, bos_token="<s>", eos_token="</s>", unk_token="<unk>", pad_token="<pad>", mask_token="<mask>")
    tokenizer.save_pretrained(tokenizer_dir)
    print(f"Tokenizer successfully saved at: {tokenizer_dir}")

Tokenizador encontrado em: /content/drive/MyDrive/Colab_LLMs/Maia_Lite/tokenizador. Carregando tokenizador existente...
Tokenizador carregado com sucesso!


In [ ]:
import os

# Set BEFORE importing torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from datasets import load_dataset, interleave_datasets
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from transformers.trainer_utils import get_last_checkpoint


# ============================================================
# MAIA LITE PRETRAINING
# Conservative configuration for Google Colab L4 / A100 / V100
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
TOKENIZER_DIR = os.path.join(PROJECT_DIR, "tokenizador")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "checkpoints_pretreino")
FINAL_MODEL_DIR = os.path.join(PROJECT_DIR, "modelo_355M_final")

MAX_LENGTH = 1024
MAX_STEPS = 300_000


# ------------------------------------------------------------
# 1. GPU detection
# ------------------------------------------------------------

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    device = torch.device("cuda")

    props = torch.cuda.get_device_properties(0)
    gpu_memory_gb = props.total_memory / (1024 ** 3)

    print(f"GPU available: {gpu_name}")
    print(f"GPU memory: {gpu_memory_gb:.1f} GB")
else:
    gpu_name = "CPU"
    device = torch.device("cpu")

    print(
        "WARNING: No CUDA GPU available. "
        "Training will be extremely slow."
    )


# ------------------------------------------------------------
# 2. Tokenizer
# ------------------------------------------------------------

tokenizer = GPT2TokenizerFast.from_pretrained(
    TOKENIZER_DIR,
    local_files_only=True,
)

# GPT-style causal LM normally uses EOS as padding token.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer vocabulary: {tokenizer.vocab_size:,}")
print(f"EOS token: {tokenizer.eos_token_id}")
print(f"PAD token: {tokenizer.pad_token_id}")


# ------------------------------------------------------------
# 3. Find latest checkpoint
# ------------------------------------------------------------

last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

if last_checkpoint is not None:
    print(f"\nCheckpoint found: {last_checkpoint}")

    model = GPT2LMHeadModel.from_pretrained(
        last_checkpoint,
        local_files_only=True,
    )

else:
    print("\nNo checkpoint found. Creating Maia Lite from scratch.")

    config = GPT2Config(
        vocab_size=tokenizer.vocab_size,
        n_positions=MAX_LENGTH,
        n_ctx=MAX_LENGTH,
        n_embd=1024,
        n_layer=24,
        n_head=16,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=False,
        tie_word_embeddings=True,
    )

    model = GPT2LMHeadModel(config)


# Important when using gradient checkpointing
model.config.use_cache = False

model.to(device)

print(f"Model parameters: {model.num_parameters():,}")


# ------------------------------------------------------------
# 4. Verify tied embeddings
# ------------------------------------------------------------

try:
    tied = (
        model.transformer.wte.weight.data_ptr()
        == model.lm_head.weight.data_ptr()
    )

    print(f"Input/output embeddings tied: {tied}")

except Exception as exc:
    print(f"Could not verify tied embeddings: {exc}")


# ------------------------------------------------------------
# 5. Gradient checkpointing
#
# KEEP ENABLED.
# Maia Lite previously experienced CUDA OOM on Colab.
# Reliability is more important than a small speed gain.
# ------------------------------------------------------------

model.gradient_checkpointing_enable()

print("Gradient checkpointing: ENABLED")


# ------------------------------------------------------------
# 6. Wikipedia streaming datasets
# ------------------------------------------------------------

print("\nLoading Wikipedia streams...")

wiki_en = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train",
    streaming=True,
)

wiki_pt = load_dataset(
    "wikimedia/wikipedia",
    "20231101.pt",
    split="train",
    streaming=True,
)

wiki_es = load_dataset(
    "wikimedia/wikipedia",
    "20231101.es",
    split="train",
    streaming=True,
)

dataset_mixed = interleave_datasets(
    [wiki_en, wiki_pt, wiki_es],
    probabilities=[0.50, 0.25, 0.25],
    seed=42,
)


# ------------------------------------------------------------
# 7. Tokenization
#
# IMPORTANT:
# We deliberately keep the current tokenization strategy while
# resuming this pretraining run.
#
# Changing to document packing halfway through the experiment
# changes the training-data distribution.
#
# Packing should be implemented and benchmarked separately for
# the next Maia pretraining run.
# ------------------------------------------------------------

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=True,
    )


tokenized_dataset = dataset_mixed.map(
    tokenize_function,
    batched=True,
    remove_columns=["id", "url", "title", "text"],
)


# ------------------------------------------------------------
# 8. Causal language-model collator
# ------------------------------------------------------------

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)


# ------------------------------------------------------------
# 9. Conservative GPU profile
# ------------------------------------------------------------

batch_size = 1
gradient_accumulation = 16
use_bf16 = False
use_fp16 = True


if "A100" in gpu_name:

    # Still conservative because the same notebook may receive
    # different A100 memory configurations.
    batch_size = 8
    gradient_accumulation = 4

    use_bf16 = True
    use_fp16 = False


elif "L4" in gpu_name:

    # Known-safe configuration from the current Maia run.
    batch_size = 4
    gradient_accumulation = 8

    use_bf16 = True
    use_fp16 = False


elif "V100" in gpu_name:

    batch_size = 4
    gradient_accumulation = 8

    use_bf16 = False
    use_fp16 = True


elif "T4" in gpu_name:

    batch_size = 1
    gradient_accumulation = 16

    use_bf16 = False
    use_fp16 = True


effective_batch = batch_size * gradient_accumulation

print("\nTraining configuration")
print("----------------------")
print(f"GPU:                   {gpu_name}")
print(f"Micro batch:           {batch_size}")
print(f"Gradient accumulation: {gradient_accumulation}")
print(f"Effective batch:       {effective_batch}")
print(f"Context length:        {MAX_LENGTH}")
print(f"BF16:                  {use_bf16}")
print(f"FP16:                  {use_fp16}")


# ------------------------------------------------------------
# 10. Training arguments
# ------------------------------------------------------------

training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    # Total training target
    max_steps=MAX_STEPS,

    # Memory-safe batching
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation,

    # Precision
    bf16=use_bf16,
    fp16=use_fp16,

    # Optimization
    learning_rate=4e-4,
    weight_decay=0.01,

    # Explicit scheduler
    lr_scheduler_type="linear",
    warmup_steps=3000,

    # Logging
    logging_steps=100,

    # Checkpoints
    save_steps=2000,
    save_total_limit=2,

    # Streaming dataset
    dataloader_num_workers=2,

    # IMPORTANT:
    # Keep this behavior because restarting a huge streaming
    # dataset and skipping tens of thousands of batches can take
    # a very long time in Colab.
    ignore_data_skip=True,

    # Avoid unnecessary integrations
    report_to="none",

    # Reduces CPU -> GPU transfer overhead
    dataloader_pin_memory=True,

    # Let Trainer remove unused dataset columns
    remove_unused_columns=True,
)


# ------------------------------------------------------------
# 11. Trainer
# ------------------------------------------------------------

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)


# ------------------------------------------------------------
# 12. CUDA diagnostics before training
# ------------------------------------------------------------

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)

    print("\nCUDA memory before training")
    print("---------------------------")
    print(f"Allocated: {allocated:.2f} GB")
    print(f"Reserved:  {reserved:.2f} GB")


# ------------------------------------------------------------
# 13. Resume training
# ------------------------------------------------------------

print("\nStarting Maia Lite pretraining...")

if last_checkpoint is not None:

    print(f"Resuming Trainer state from:")
    print(last_checkpoint)

    trainer.train(
        resume_from_checkpoint=last_checkpoint
    )

else:

    trainer.train()


# ------------------------------------------------------------
# 14. Save final consolidated model
# ------------------------------------------------------------

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("\n========================================")
print("MAIA LITE PRETRAINING COMPLETED")
print("========================================")
print(f"Final model saved to: {FINAL_MODEL_DIR}")

GPU available: NVIDIA L4
GPU memory: 22.0 GB
Tokenizer vocabulary: 50,257
EOS token: 2
PAD token: 1

Checkpoint found: /content/drive/MyDrive/Colab_LLMs/Maia_Lite/checkpoints_pretreino/checkpoint-22000


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Model parameters: 354,823,168
Input/output embeddings tied: True
Gradient checkpointing: ENABLED

Loading Wikipedia streams...


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss
22100,3.256821
22200,3.262606
22300,3.228669
22400,3.131976
22500,3.065855
22600,3.087887
22700,3.114410
22800,3.055420
22900,3.132237
23000,3.170550
